In [ ]:
# ============================================================
# Full-data preprocessing .
# Streams HF dataset, writes stratified Train/Test Parquet shards,
# and saves class weights + manifest for any downstream model.
# ============================================================

from datasets import load_dataset
import polars as pl
from pathlib import Path
import json
import random

# -------------------- FIXED CONFIG (edit here if needed) --------------------
ROOT = Path.cwd()
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

LABEL_COL   = "click"   # target column
SPLIT_NAME  = "train"   # HF split to stream: "train" / "validation" / "test"
TRAIN_RATIO = 0.80      # proportion routed to train (approx stratified)
SHARD_ROWS  = 500_000   # rows per parquet shard
SEED        = 42        # reproducibility for routing decisions

random.seed(SEED)

# -------------------- helpers --------------------
def flush_buffer(rows_buf, split_name, shard_idx):
    """Write buffered rows into a Parquet shard and clear the buffer."""
    if not rows_buf:
        return shard_idx, None, 0
    df_pl = pl.DataFrame(rows_buf).with_columns(
        pl.col(LABEL_COL).cast(pl.Int8, strict=False)
    )
    out_path = DATA_PROCESSED / f"{split_name}_{shard_idx:03d}.parquet"
    df_pl.write_parquet(out_path)
    n_rows = len(rows_buf)
    rows_buf.clear()
    return shard_idx + 1, str(out_path), n_rows

def route_to_train(y, n_train_pos, n_train_neg, n_test_pos, n_test_neg):
    """Approximate stratified routing keeping per-class train ratio near TRAIN_RATIO."""
    if y == 1:
        total_pos = n_train_pos + n_test_pos
        # if no positives yet → send to train; else keep ratio near TRAIN_RATIO
        return (total_pos == 0) or (n_train_pos / max(1, total_pos) < TRAIN_RATIO)
    else:
        total_neg = n_train_neg + n_test_neg
        return (total_neg == 0) or (n_train_neg / max(1, total_neg) < TRAIN_RATIO)

# -------------------- stream & shard --------------------
print(f"[INFO] Streaming Hugging Face dataset: BadrAbu/CTR_Prediction split='{SPLIT_NAME}'")
ds = load_dataset("BadrAbu/CTR_Prediction", split=SPLIT_NAME, streaming=True)

buf_train, buf_test = [], []
train_shards, test_shards = [], []

# counters for stratified routing
n_train_pos = n_train_neg = 0
n_test_pos  = n_test_neg  = 0

shard_idx_train = 0
shard_idx_test  = 0
rows_since_flush = 0

for row in ds:
    y = int(row[LABEL_COL])
    if route_to_train(y, n_train_pos, n_train_neg, n_test_pos, n_test_neg):
        buf_train.append(row)
        if y == 1: n_train_pos += 1
        else:      n_train_neg += 1
    else:
        buf_test.append(row)
        if y == 1: n_test_pos += 1
        else:      n_test_neg += 1

    rows_since_flush += 1
    if rows_since_flush >= SHARD_ROWS:
        shard_idx_train, path_t, n_t = flush_buffer(buf_train, "train", shard_idx_train)
        if path_t: train_shards.append({"path": path_t, "rows": n_t})
        shard_idx_test,  path_v, n_v = flush_buffer(buf_test,  "test",  shard_idx_test)
        if path_v: test_shards.append({"path": path_v, "rows": n_v})
        rows_since_flush = 0

# final flush
shard_idx_train, path_t, n_t = flush_buffer(buf_train, "train", shard_idx_train)
if path_t: train_shards.append({"path": path_t, "rows": n_t})
shard_idx_test,  path_v, n_v = flush_buffer(buf_test,  "test",  shard_idx_test)
if path_v: test_shards.append({"path": path_v, "rows": n_v})

# -------------------- class weights (for imbalanced data) --------------------
total_pos = n_train_pos + n_test_pos
total_neg = n_train_neg + n_test_neg
total_all = total_pos + total_neg

# simple balanced weights: N / (2 * N_class)
class_weights = {
    0: float(total_all / (2.0 * max(1, total_neg))),
    1: float(total_all / (2.0 * max(1, total_pos))),
}

# -------------------- manifest --------------------
manifest = {
    "label_col": LABEL_COL,
    "split_streamed": SPLIT_NAME,
    "train_ratio": TRAIN_RATIO,
    "seed": SEED,
    "shard_rows": SHARD_ROWS,
    "counts": {
        "train": {"neg": n_train_neg, "pos": n_train_pos, "total": n_train_neg + n_train_pos},
        "test":  {"neg": n_test_neg,  "pos": n_test_pos,  "total": n_test_neg + n_test_pos},
        "overall": {"neg": total_neg, "pos": total_pos, "total": total_all},
    },
    "class_weights": class_weights,
    "train_shards": train_shards,
    "test_shards": test_shards,
}

with open(DATA_PROCESSED / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

with open(DATA_PROCESSED / "class_weights.json", "w") as f:
    json.dump(class_weights, f, indent=2)

print("\n✅ DONE")
print(f"  Train shards: {len(train_shards)} | Test shards: {len(test_shards)}")
print(f"  Counts → train(pos={n_train_pos}, neg={n_train_neg}) | test(pos={n_test_pos}, neg={n_test_neg})")
print(f"  Saved manifest:      {DATA_PROCESSED / 'manifest.json'}")
print(f"  Saved class weights: {DATA_PROCESSED / 'class_weights.json'}")
print(f"  Parquet output dir:  {DATA_PROCESSED.resolve()}")
